# 08. 감성 분석
## 주제: 배달 음식 서비스 리뷰
## 2021040032 임경택

### 0. 환경과 재현성

In [5]:
import re
import sys
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
LABELS = [0, 1]  # 0: 부정, 1: 긍정
LABEL_NAMES = ['부정', '긍정']
print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__, '| sklearn:', sklearn.__version__)

Python: 3.14.6
pandas: 3.0.3 | sklearn: 1.9.0


### 1. 데이터 이해와 전처리
- AI를 활용해 조건에 따라 데이터를 생성했습니다.

In [7]:
positive_reviews = [
    "음식이 따뜻하게 도착해서 정말 맛있게 먹었어요",
    "배달 시간이 예상보다 훨씬 빨라서 만족합니다",
    "포장이 꼼꼼해서 국물이 하나도 안 샜어요",
    "치킨이 바삭하고 양도 넉넉해서 최고였어요",
    "기사님이 친절하게 문 앞까지 조심히 놓아주셨어요",
    "떡볶이 맛이 매장에서 먹는 것과 똑같이 맛있었어요",
    "리뷰 이벤트 서비스까지 챙겨주셔서 감사합니다",
    "피자가 식지 않고 치즈가 쭉쭉 늘어나서 좋았어요",
    "주문한 지 20분 만에 도착해서 놀랐어요 빠르고 좋네요",
    "짜장면 면이 하나도 안 불어서 신기했어요 맛있습니다",
    "요청사항을 정확하게 지켜주셔서 너무 만족해요",
    "재료가 신선하고 양이 많아서 재주문 의사 있습니다",
    "용기가 깔끔하고 밀봉이 잘 되어 있어서 위생적이었어요",
    "비 오는 날인데도 음식이 젖지 않게 잘 배달해주셨어요",
    "양념이 적당해서 아이들도 맛있게 먹었어요",
    "가격 대비 양이 푸짐해서 가성비 최고예요",
    "배달 예상 시간이 정확해서 기다리기 편했어요",
    "회가 신선하고 얼음팩까지 챙겨주셔서 감동이었어요",
    "ㅋㅋㅋ 진짜 맛있어서 순식간에 다 먹었어요 또 시킬게요",          # [구어]
    "존맛탱 ㅠㅠ 이 집 없었으면 어쩔 뻔 ㅋㅋ",                        # [구어]
    "국물이 뜨끈하고 진해서 해장으로 딱이었어요 👍",                  # [구어]
    "배달이 조금 늦었지만 음식이 너무 맛있어서 다 용서됩니다",        # [혼합]
    "가격은 좀 비싸지만 맛과 양을 생각하면 충분히 만족해요",          # [혼합]
    "포장은 평범했는데 맛은 기대 이상이었어요",                        # [혼합]
    "단골인데 언제 시켜도 맛이 한결같아서 좋아요",
    "소스를 넉넉하게 넣어주셔서 끝까지 맛있게 먹었어요",
    "기사님이 도착 전에 미리 연락주셔서 편했어요",
    "튀김이 눅눅하지 않고 바삭해서 놀랐어요",
    "첫 주문이었는데 실패 없이 대만족입니다",
    "음식 온도도 맛도 완벽해서 별 다섯 개 드려요",
]

negative_reviews = [
    "음식이 다 식어서 도착해서 너무 실망했어요",
    "배달이 한 시간 넘게 지연되어서 화가 났어요",
    "포장이 엉망이라 국물이 다 새서 봉투가 젖었어요",
    "치킨이 눅눅하고 양도 너무 적었어요",
    "기사님이 불친절하고 음식을 던지듯이 놓고 가셨어요",
    "주문한 메뉴와 다른 음식이 왔어요",
    "짜장면이 완전히 불어서 먹을 수가 없었어요",
    "요청사항을 전혀 지키지 않아서 짜증났어요",
    "음식에서 머리카락이 나와서 위생이 의심됩니다",
    "맛이 너무 짜고 기름져서 반도 못 먹었어요",
    "가격에 비해 양이 형편없이 적어서 다시는 안 시킵니다",
    "피자가 한쪽으로 쏠려서 토핑이 다 떨어져 있었어요",
    "예상 시간보다 40분이나 늦게 와서 최악이었어요",
    "고기가 질기고 냄새가 나서 별로였어요",
    "배달 기사님이 연락도 없이 엉뚱한 곳에 두고 가셨어요",
    "소스가 빠져 있어서 맛없게 먹었어요",
    "용기가 찌그러져 있고 밀봉도 안 되어 있었어요",
    "와 한 시간 반 만에 오다니 정말 빠르네요 감사합니다^^",          # [반어]
    "차갑게 식은 국밥이라니 여름 메뉴로 딱이네요",                    # [반어]
    "양이 어찌나 많은지 한입에 다 먹었네요 대단합니다",                # [반어]
    "아 진짜 너무하네요 ㅠㅠ 다 식어서 왔어요",                        # [구어]
    "ㅡㅡ 두 번 다시 안 시킴 맛도 없고 늦고",                          # [구어]
    "맛은 괜찮았는데 배달이 너무 늦어서 다 식어버렸어요",              # [혼합]
    "기사님은 친절했지만 음식 상태가 너무 별로였어요",                # [혼합]
    "양은 많은데 맛이 없어서 결국 남겼어요",                          # [혼합]
    "튀김옷만 두껍고 속은 거의 비어 있었어요",
    "리뷰 이벤트 서비스를 요청했는데 빠져 있었어요",
    "배달 앱에는 도착이라고 떴는데 음식이 오지 않았어요",
    "회가 미지근하게 와서 신선도가 떨어졌어요",
    "비닐 포장이 뜯겨 있어서 누가 손댄 것 같아 불쾌했어요",
]
raw_df = pd.DataFrame({
    'text': positive_reviews + negative_reviews,
    'label': [1] * len(positive_reviews) + [0] * len(negative_reviews)
})

def normalize_text(text):
    return re.sub(r'\s+', ' ', str(text)).strip() # 반복된 공백만 공백 하나로 바꾸고, 맨앞과 맨뒤의 공백만 제거.

df = raw_df.dropna(subset=['text', 'label']).copy() # text, label에 결측치 없는 행만 새로운 데이터프레임으로 복사.
df['text'] = df['text'].map(normalize_text) # text에 대해 간단한 정규화 진행
df = df[df['text'].str.len() > 0].copy() # text의 길이가 0보다 큰 행만 새로운 데이터프레임으로 복사.

# 실제 데이터는 같은 텍스트의 상충 라벨을 별도로 검토해야 한다.
assert df.groupby('text')['label'].nunique().max() == 1, '상충 라벨을 먼저 검토하세요.' # 한 텍스트에 대해 라벨이 2가지 달린 경우 없는지 체크(조건식이 False라면 AssertionError의 메시지가 출력).
assert set(df['label']) == {0, 1}

df = df.drop_duplicates(subset=['text']).reset_index(drop=True) # 동일한 텍스트가 있다면 제거. 인덱스 다시 정렬.
df['review_id'] = np.arange(len(df)) # 0부터 len(df)-1번까지

print('정제 전/후:', len(raw_df), len(df))
display(df.groupby('label').size().rename(index={0:'부정', 1:'긍정'}).to_frame('count')) # label 기준으로 그룹화하고, 고유값별 행 개수를 센 다음에, 인덱스를 숫자에서 단어로 바꾼 뒤, 개수 열의 이름을 'count'로 지정해 df로 변환.
display(df.sample(6, random_state=SEED)) # 랜덤하게 6개의 행 출력

정제 전/후: 60 60


,count
label,
부정,30
긍정,30


,text,label,review_id
0,음식이 따뜻하게 도착해서 정말 맛있게 먹었어요,1,0
5,떡볶이 맛이 매장에서 먹는 것과 똑같이 맛있었어요,1,5
36,짜장면이 완전히 불어서 먹을 수가 없었어요,0,36
45,소스가 빠져 있어서 맛없게 먹었어요,0,45
13,비 오는 날인데도 음식이 젖지 않게 잘 배달해주셨어요,1,13
54,양은 많은데 맛이 없어서 결국 남겼어요,0,54


### 2. 규칙 기반 baseline

In [13]:
LEXICON = {'따뜻': 1, '맛있': 2, '빨라': 2, '만족': 2, '꼼꼼': 1, '바삭': 1, '넉넉': 1, '푸짐': 1, '최고': 2, '친절': 1, '조심': 1, '좋': 2, '신선': 1, '감동': 2, '깔끔': 1, '정확': 1, '완벽': 2,  '편': 1, 
           '식': -1, '맛없': -2, '늦': -2, '실망': -2, '엉망': -1, '눅눅': -1, '적': -1, '남겼': -1, '짜증': -2, '불친절': -1, '빠져': -1, '화': -2, '미지근': -1, '최악': -2, '지연': -1, '질기고': -1, '짜': -2, '불쾌': -1}

def lexicon_score(text):
    return sum(weight for term, weight in LEXICON.items() if term in text) # 감정사전의 모든 용어 중에서, text에 등장한 용어의 가중치를 더함.

def score_to_label(score):
    return '긍정' if score > 0 else ('부정' if score < 0 else '판단보류')

rule_examples = ['정말 맛있다', '정말 맛없다', '만족하지 않았다',
                 '음식은 좀 짜지만 따뜻했다', '기사님과 카드로 결제했다']
rule_table = pd.DataFrame({'text': rule_examples})
rule_table['score'] = rule_table['text'].map(lexicon_score)
rule_table['prediction'] = rule_table['score'].map(score_to_label)
display(rule_table)

,text,score,prediction
0,정말 맛있다,2,긍정
1,정말 맛없다,-2,부정
2,만족하지 않았다,2,긍정
3,음식은 좀 짜지만 따뜻했다,-2,부정
4,기사님과 카드로 결제했다,0,판단보류


#### 한계
- '않'을 반영하지 못했습니다.
- 대조 부분에서 실제로 강조되는 것이 뒷부분이라는 것을 반영하지 못했습니다.
- 형태소 분석을 진행하지 않아, '음식은'에서 '식었다'의 어근인 '식'이 있다고 잘못 판단하여 점수가 잘못 산정되었습니다.

In [16]:
def improved_rule_score(text):
    score = lexicon_score(text)
    if '않' or '못 ' in text:
        score = score * (-1)
    if '만 ' in text:
        score = score * (-1)

    return score

rule_table['improved_score'] = rule_table['text'].map(improved_rule_score)
rule_table['improved_prediction'] = rule_table['improved_score'].map(score_to_label)
display(rule_table)
print('수정 완료 확인:', improved_rule_score('만족하지 않았다') < 0)

,text,score,prediction,improved_score,improved_prediction
0,정말 맛있다,2,긍정,-2,부정
1,정말 맛없다,-2,부정,2,긍정
2,만족하지 않았다,2,긍정,-2,부정
3,음식은 좀 짜지만 따뜻했다,-2,부정,-2,부정
4,기사님과 카드로 결제했다,0,판단보류,0,판단보류


수정 완료 확인: True


### 3. 텍스트 상태로 train / validation / test 분할

In [20]:
train_df, heldout_df = train_test_split(df, test_size=0.4, stratify=df['label'], random_state=SEED) # 6:4
valid_df, test_df = train_test_split(heldout_df, test_size=0.5, stratify=heldout_df['label'], random_state=SEED) # 2:2

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

for a, b in [(train_df, valid_df), (train_df, test_df), (valid_df, test_df)]: # 2개씩 짝지어서 3번 비교
    assert set(a.review_id).isdisjoint(set(b.review_id)) # 리뷰 아이디가 겹치지 않는지
    assert set(a.text).isdisjoint(set(b.text)) # 텍스트가 겹치지 않는지

display(pd.DataFrame({name: part_df['label'].value_counts().sort_index()
                    for name, part_df in [('train', train_df), ('validation', valid_df), ('test', test_df)]}))

,train,validation,test
label,,,
0,18,6,6
1,18,6,6


### 4. NB / LR 비교

In [21]:
def make_pipeline(classifier):
    return Pipeline([
        ('tfidf', TfidfVectorizer(tokenizer=str.split, token_pattern=None,
                                  lowercase=False, ngram_range=(1, 2), min_df=1)),
        ('clf', classifier)
    ])

models = {
    'NB': make_pipeline(MultinomialNB(alpha=1.0)),
    'LR': make_pipeline(LogisticRegression(C=1.0, max_iter=1000, random_state=SEED))
}

def evaluate(y_true, y_pred):
    return {'accuracy': accuracy_score(y_true, y_pred),
            'precision_positive': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
            'recall_positive': recall_score(y_true, y_pred, pos_label=1, zero_division=0),
            'f1_positive': f1_score(y_true, y_pred, pos_label=1, zero_division=0),
            'macro_f1': f1_score(y_true, y_pred, average='macro', labels=LABELS, zero_division=0)}

validation_predictions = {}
validation_rows = []
for name, model in models.items():
    model.fit(train_df.text, train_df.label) # train_set으로 학습
    pred = model.predict(valid_df.text) # valid_set으로 예측
    validation_predictions[name] = pred
    validation_rows.append({'model': name, **evaluate(valid_df.label, pred)}) # 성능 확인(언패킹 이용해 평가 데이터를 리스트에 추가)
validation_results = pd.DataFrame(validation_rows).set_index('model')
display(validation_results.round(3))

,accuracy,precision_positive,recall_positive,f1_positive,macro_f1
model,,,,,
NB,0.5,0.5,0.333,0.400,0.486
LR,0.5,0.5,0.667,0.571,0.486


#### 규칙 baseline 확인

In [24]:
rule_scores = valid_df.text.map(lexicon_score)
covered = rule_scores.ne(0) # 0과 다른지
rule_pred = (rule_scores[covered] > 0).astype(int)
print('판단 가능 비율(coverage):', round(float(covered.mean()), 3))
if covered.any():
    print('판단 가능한 문장에서만 정확도:', round(accuracy_score(valid_df.loc[covered, 'label'], rule_pred), 3))
print('판단보류 수:', int((~covered).sum()))

판단 가능 비율(coverage): 0.667
판단 가능한 문장에서만 정확도: 1.0
판단보류 수: 4


### 5. Validation 오류 분석과 특징 해석

In [31]:
best_name = validation_results.sort_values('macro_f1', ascending=False, kind='stable').index[0]
selected_model = models[best_name]
valid_analysis = valid_df[['review_id', 'text', 'label']].copy()
valid_analysis['prediction'] = validation_predictions[best_name]
valid_analysis['error_type'] = ''

errors = valid_analysis[valid_analysis['label'] != valid_analysis['prediction']].copy()
errors['error_type'] = ['단어 조합', '부정어', '부정어', '라벨링 오류', '라벨링 오류', '용어 미탐']
errors['details'] = [
    "공백 토큰화 진행 시 '하나도'가 '안 불어서'와 연결되지 않고 혼자 가중치 높은 부정적인 단어로 들어가서 부정으로 판단한 것으로 보입니다.",
    "'실패'가 부정어 '없이'와 연결되지 않아 부정으로 판단한 것으로 보입니다.",
    "부정어 '안'이 '샜어요'와 연결되지 않아 부정으로 판단한 것으로 보입니다.",
    "빠져 있었어요'가 전체적인 문맥에서 해석되지 않고 긍정적인 단어 조합으로 들어가서 긍정으로 판단한 것으로 보입니다",
    "'눅눅'이라는 표현이 긍정적인 단어로 들어가서 긍정으로 판단한 것으로 보입니다.",
    "'존맛탱'이라는 단어에 대한 판단이 잘 이뤄지지 않아 부정으로 판단한 것으로 보입니다."
]
print('validation 선택 모델:', best_name, '(동률이면 사전 순서 NB 우선)')
display(errors)


validation 선택 모델: NB (동률이면 사전 순서 NB 우선)


,review_id,text,label,prediction,error_type,details
0,9,짜장면 면이 하나도 안 불어서 신기했어요 맛있습니다,1,0,단어 조합,공백 토큰화 진행 시 '하나도'가 '안 불어서'와 연결되지 않고 혼자 가중치 높은 ...
2,28,첫 주문이었는데 실패 없이 대만족입니다,1,0,부정어,'실패'가 부정어 '없이'와 연결되지 않아 부정으로 판단한 것으로 보입니다.
5,2,포장이 꼼꼼해서 국물이 하나도 안 샜어요,1,0,부정어,부정어 '안'이 '샜어요'와 연결되지 않아 부정으로 판단한 것으로 보입니다.
6,56,리뷰 이벤트 서비스를 요청했는데 빠져 있었어요,0,1,라벨링 오류,빠져 있었어요'가 전체적인 문맥에서 해석되지 않고 긍정적인 단어 조합으로 들어가서 ...
9,33,치킨이 눅눅하고 양도 너무 적었어요,0,1,라벨링 오류,'눅눅'이라는 표현이 긍정적인 단어로 들어가서 긍정으로 판단한 것으로 보입니다.
11,19,존맛탱 ㅠㅠ 이 집 없었으면 어쩔 뻔 ㅋㅋ,1,0,용어 미탐,'존맛탱'이라는 단어에 대한 판단이 잘 이뤄지지 않아 부정으로 판단한 것으로 보입니다.


In [27]:
lr_classifier = models['LR'].named_steps['clf']
if isinstance(lr_classifier, LogisticRegression):
    assert list(lr_classifier.classes_) == [0, 1]
    weights = pd.DataFrame({'feature': models['LR'].named_steps['tfidf'].get_feature_names_out(),
                            'weight': lr_classifier.coef_[0]})
    display(weights.nsmallest(8, 'weight'))
    display(weights.nlargest(8, 'weight'))

,feature,weight
67,다,-0.257483
260,와서,-0.234473
171,별로였어요,-0.232540
297,있었어요,-0.206011
198,시간,-0.189936
360,한,-0.189936
361,한 시간,-0.189936
44,너무,-0.157995


,feature,weight
339,챙겨주셔서,0.240221
354,편했어요,0.220500
203,시간이,0.219385
114,만족해요,0.217938
298,잘,0.199708
46,너무 만족해요,0.140935
266,요청사항을,0.140935
267,요청사항을 정확하게,0.140935


### 6. 선택 확정

In [40]:
from sklearn.base import clone

if globals().get('TEST_EVALUATED', False):
     print('이미 test 평가를 완료했습니다. 저장한 결과를 참고하세요.')
else:
    final_model = clone(models[best_name])
    devlopment_df = pd.concat([train_df, valid_df], ignore_index=True)
    final_model.fit(development_df.text, development_df.label)
    test_pred = final_model.predict(test_df.text)
    test_metrics = evaluate(test_df.label, test_pred)
    test_cm = confusion_matrix(test_df.label, test_pred)
    TEST_EVALUATED = True
    display(pd.DataFrame([test_metrics], index=[best_name]).round(3))
    print(classification_report(test_df.label, test_pred, labels=LABELS,
                                target_names=LABEL_NAMES, zero_division=0))
    display(pd.DataFrame(test_cm, index=['실제 부정', '실제 긍정'], columns=['예측 부정', '예측 긍정']))
    print('행=실제, 열=예측. 긍정 기준: 좌상 TN, 우상 FP, 좌하 FN, 우하 TP')

,accuracy,precision_positive,recall_positive,f1_positive,macro_f1
NB,0.5,0.5,0.5,0.5,0.5


              precision    recall  f1-score   support

          부정       0.50      0.50      0.50         6
          긍정       0.50      0.50      0.50         6

    accuracy                           0.50        12
   macro avg       0.50      0.50      0.50        12
weighted avg       0.50      0.50      0.50        12



,예측 부정,예측 긍정
실제 부정,3,3
실제 긍정,3,3


행=실제, 열=예측. 긍정 기준: 좌상 TN, 우상 FP, 좌하 FN, 우하 TP


### 7. 새로운 리뷰와 상품 도메인 탐색

In [41]:
transfer_sentences = [
    # 전이 도메인: 택배·배송 서비스
    "택배 상자가 찌그러지고 안에 있던 그릇이 파손되어 왔어요",
    "송장 번호를 조회해도 계속 상차 대기라서 답답합니다",
    "기사님이 반품 회수를 빠르게 처리해주셔서 편했어요",
    "주문 다음 날 새벽에 받아서 깜짝 놀랐어요 포장도 튼튼해요",
    # 주 도메인: 배달 음식 서비스
    "떡볶이가 식지 않고 따뜻하게 와서 맛있었어요",
    "배달이 너무 늦어서 음식이 다 불어버렸어요",
    "맛은 그럭저럭인데 양이 너무 적어요",
]

new_reviews = pd.DataFrame({'text': transfer_sentences, 'domain': ['택배배송'] * 4 + ['배달음식'] * 3})
if globals().get('TEST_EVALUATED', False):
    new_reviews['prediction'] = final_model.predict(new_reviews.text)
    probabilities = final_model.predict_proba(new_reviews.text)
    for col, label in enumerate(final_model.classes_):
        new_reviews['p_' + LABEL_NAMES[int(label)]] = probabilities[:, col]
    display(new_reviews)

,text,domain,prediction,p_부정,p_긍정
0,택배 상자가 찌그러지고 안에 있던 그릇이 파손되어 왔어요,택배배송,0,0.612746,0.387254
1,송장 번호를 조회해도 계속 상차 대기라서 답답합니다,택배배송,0,0.500000,0.500000
2,기사님이 반품 회수를 빠르게 처리해주셔서 편했어요,택배배송,1,0.422040,0.577960
3,주문 다음 날 새벽에 받아서 깜짝 놀랐어요 포장도 튼튼해요,택배배송,1,0.392792,0.607208
4,떡볶이가 식지 않고 따뜻하게 와서 맛있었어요,배달음식,1,0.395357,0.604643
5,배달이 너무 늦어서 음식이 다 불어버렸어요,배달음식,0,0.682175,0.317825
6,맛은 그럭저럭인데 양이 너무 적어요,배달음식,0,0.604567,0.395433


### 8. 해석 보고서
- 1번 문장에서 '상차 대기'가 미등록 어휘로 사라졌을 것으로 보입니다. '답답합니다'로도 부정을 체크할 수 있어야 했지만, '상차 대기'까지 있었다면 부정 확률이 좀 더 올라갔을 것으로 판단됩니다.